### The Training Problem

The quantized n-bit weights **cannot be trained directly:**
- They're n-bit integers, not continuous floats
- Can't compute gradients for discrete integers
- If you dequantize to FP16, train, then re-quantize → massive information loss (apart from the fact that it won't fit in memory as was the bottleneck originally)

### 1. The Solution: QLoRA (Quantized LoRA)

Keep the n-bit weights **frozen**, train only **low-rank adapters**:

Frozen n-bit base weights
       ↓ (dequantized for forward pass)
   [Output of base layer]
       ↓
   LoRA adapters (trainable, FP16)
       ↓
   [Combined output with gradient flow]



**During training:**

1. Forward pass: dequantize n-bit weights layer-by-layer (same as inference)
2. Compute: `output = base_output + LoRA_A @ LoRA_B`
3. Backward: gradients only flow into `LoRA_A` and `LoRA_B`, not the n-bit weights
4. Update: only LoRA parameters get trained (tiny, like 1-2% of full model)

### What This Means

**BitsAndBytes alone:** ❌ Can't finetune  
**BitsAndBytes + LoRA:** ✅ Can finetune efficiently

**Your notebook likely has:**

```python
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16,  # rank
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    ...
)

peft_model = get_peft_model(model_4b, lora_config)
# Now you can train peft_model!
```

The n-bit weights stay frozen in memory, you only compute gradients for the LoRA adapters.

### 2. How Does LoRA Work?

Two things:
1. Singular Vector Decomposition (roughly, we don't actually break down the weights into two)
2. Choosing a small number for rank where rank is (n, rank) * (rank, m) a.k.a the facing dimension.

Instead of using a big matrix as adapter (which would have to be the same size as the original layer that was quantized) to store the changes (the updates), we’ll replace it with two small matrices. 
Decomposing matrices inevitably leads to losses. The smaller the two resulting matrices are, the worse the approximation is. But we’re not doing that. Instead, we’re starting with two small matrices (let’s call them A and B) which will have their weights updated during training. If we multiply them, we’ll get a full-sized matrix. This resulting matrix would represent the accumulated updates to the, now quantized and frozen, weights. In reality, we don’t need to compute this matrix; we can forward our inputs through the two small matrices in sequence and achieve the same output.


Even though the matrix is full-sized, most of its values are just linear combinations of
other values. The rank shows you the real dimensions of the matrix—or basically, how much redundancy is in there:

- A high-ranking value (that is, one close to the actual number of rows or columns) means there’s little to no redundancy.
- A low-rank matrix implies there’s a lot of redundancy, and the large matrix can be easily represented by two smaller ones.


Now, you know why it’s called a low-rank adapter. We’re using two small matrices to produce a low-rank big matrix (representing the updates) that matches the size of the original layer.
While low-rank matrices may be sufficient for fine-tuning, they can **severely impede** the learning ability of a model being trained from scratch. The low-rank nature of the matrices acts as a constrain, **limiting the model’s capacity to explore and learn complex relationships, effectively functioning as if it were a much smaller model**.
A pre-trained model, however, has already learned these relationships. Fine-tuning, as the name suggests, introduces only small changes to the underlying model, which can be effectively represented by low-rank matrices

### 2.1 Working out a code example

This is our original full layer (as it would be in a model, among many many layers)

```python 
base_layer = nn.Linear(1024, 1024, bias=False)
```

Now we make smaller matrices for it that multiple upto become the same size.

```python
torch.manual_seed(11)
r = 8
layer_A = nn.Linear(base_layer.in_features, r, bias=False)
layer_B = nn.Linear(r, base_layer.out_features, bias=False)
```

Now the forward pass should look like this:
```python
torch.manual_seed(19)
batch = torch.randn(1, 1024)
batch @ (base_layer.weight.data + layer_B.weight @ layer_A.weight).
```

With the distributive property of matrix multiplication, we can split this into two passes: one forward pass through the base layer, and another through the resulting low-rank matrix just like p . (q+r) = p.q + p.r

```python
regular_output = batch @ base_layer.weight.data.T
additional_output = batch @ (layer_B.weight @ layer_A.weight).T
regular_output, additional_output
```

Another addition property allows us to chain the operation one on the other for `additional_output`. The whole forward pass would look something like below:
```python
regular_output = base_layer(batch)
out_A = layer_A(batch)
additional_output = layer_B(out_A)
alpha = 2*r
output = regular_output + (alpha / r) * additional_output
```

And we have a forward pass with LoRA!!
